# 183. Corrective RAG：检索错了以后怎样评估、纠错和降级？

> **面试问题：CRAG 的 Correct/Ambiguous/Incorrect 分支如何设计？knowledge strip、外部检索和重组怎样避免引入更多噪声？**

## 先给结论

CRAG 在生成前先评估召回质量：高置信正确则精炼本地文档，模糊则本地与外部补充，明显错误则触发替代检索。纠错不是无限搜网页；外部源、超时、权限、证据 strip 和最终 claim 都必须有 provenance 与预算。

## 推荐回答主线

1. 用校准后的 retrieval evaluator 聚合 passage 分数，映射 Correct/Ambiguous/Incorrect 三态。
2. 把文档拆成 knowledge strips，逐片评分过滤，再去重重组，保留 doc/span 来源。
3. 仅在策略允许时使用外部 fallback，执行域名、时间、ACL、注入与超时检查。
4. 按分支统计 Recall、支持率、外部调用、延迟和误路由，绑定 evaluator/阈值/索引版本。

## 教学实现边界

示例没有真实网络请求，外部检索是可信 mock；相关性和支持度用受控特征，重点展示纠错控制流而非复现论文模型。

## 一手资料

- [Corrective Retrieval Augmented Generation](https://arxiv.org/abs/2401.15884)
- [Retrieval-Augmented Generation](https://arxiv.org/abs/2005.11401)
- [Self-RAG](https://arxiv.org/abs/2310.11511)


In [ ]:
import hashlib  # 导入本单元需要的依赖。
import json  # 导入本单元需要的依赖。
import math  # 导入本单元需要的依赖。
import re  # 导入本单元需要的依赖。
from dataclasses import asdict, dataclass  # 导入本单元需要的依赖。
from enum import Enum  # 导入本单元需要的依赖。
from urllib.parse import urlsplit  # 导入本单元需要的依赖。

import numpy as np  # 导入本单元需要的依赖。

# 本地召回保留 retriever 分数、来源、可验证事实及同一 claim 的结构化值。
QUERY = "BitNet b1.58 的权重取值是什么？"  # 计算并保存当前步骤的中间状态。
LOCAL = [  # 计算并保存当前步骤的中间状态。
    {"id": "local-1", "score": 0.62, "text": "BitNet 使用低比特权重。b1.58 的权重是 -1、0、1。量化需配套训练。", "facts": {"ternary_weights"}, "claims": {"weight_values": (-1, 0, 1)}},  # 执行当前语句以推进本节示例。
    {"id": "local-2", "score": 0.15, "text": "普通 INT8 模型通常有 256 个整数状态。", "facts": {"int8_states"}, "claims": {"int8_states": 256}},  # 执行当前语句以推进本节示例。
]  # 执行当前语句以推进本节示例。

assert len(LOCAL) == 2  # 用受控断言验证关键不变量。
assert all(0 <= doc["score"] <= 1 for doc in LOCAL)  # 用受控断言验证关键不变量。
assert len({doc["id"] for doc in LOCAL}) == 2  # 用受控断言验证关键不变量。


## 1. Retrieval evaluator：特征先校准，再做三态路由

可组合 retriever score、reranker、query coverage 和来源质量得到相关概率。阈值间留 ambiguous 区间，避免轻微波动在 Correct/Incorrect 间跳变；每个分支都记录原因。


In [ ]:
class RetrievalState(str, Enum):  # 定义承载本节状态与行为的数据结构。
    CORRECT = "correct"  # 计算并保存当前步骤的中间状态。
    AMBIGUOUS = "ambiguous"  # 计算并保存当前步骤的中间状态。
    INCORRECT = "incorrect"  # 计算并保存当前步骤的中间状态。

def route_retrieval(probability, low=0.35, high=0.75):  # 定义本节可复用的核心函数。
    values = (probability, low, high)  # 计算并保存当前步骤的中间状态。
    if not all(isinstance(value, (int, float)) and math.isfinite(value) for value in values):  # 按当前条件选择后续控制路径。
        raise ValueError("概率与阈值必须有限")  # 遇到非法合同立即显式失败。
    if not 0 <= low < high <= 1 or not 0 <= probability <= 1:  # 按当前条件选择后续控制路径。
        raise ValueError("概率或阈值非法")  # 遇到非法合同立即显式失败。
    if probability >= high:  # 按当前条件选择后续控制路径。
        return RetrievalState.CORRECT  # 返回当前分支计算出的结果。
    if probability <= low:  # 按当前条件选择后续控制路径。
        return RetrievalState.INCORRECT  # 返回当前分支计算出的结果。
    return RetrievalState.AMBIGUOUS  # 返回当前分支计算出的结果。

# 三态及闭区间边界固定；NaN 和越界阈值 fail closed。
assert route_retrieval(0.9) is RetrievalState.CORRECT  # 用受控断言验证关键不变量。
assert route_retrieval(0.5) is RetrievalState.AMBIGUOUS  # 用受控断言验证关键不变量。
assert route_retrieval(0.1) is RetrievalState.INCORRECT  # 用受控断言验证关键不变量。
for invalid in ((float("nan"), 0.35, 0.75), (0.5, -1.0, 2.0)):  # 遍历输入元素以累积或检查结果。
    try:  # 尝试执行可能失败的受控操作。
        route_retrieval(*invalid); assert False  # 执行当前语句以推进本节示例。
    except ValueError:  # 捕获预期异常并验证失败分支。
        assert True  # 用受控断言验证关键不变量。


## 2. 聚合 passage 质量：不能让大量低质文档稀释一个强证据

整体 evaluator 可看 top-1、top-k coverage 与一致性，而非简单平均所有召回分。教学版用 noisy-or 表示至少一篇相关的概率，并对来源可信度加权。


In [ ]:
def aggregate_relevance(documents, source_reliability=1.0):  # 定义本节可复用的核心函数。
    if not isinstance(source_reliability, (int, float)) or not math.isfinite(source_reliability) or not 0 <= source_reliability <= 1:  # 按当前条件选择后续控制路径。
        raise ValueError("source reliability 必须在 [0,1]")  # 遇到非法合同立即显式失败。
    scores = []  # 计算并保存当前步骤的中间状态。
    for document in documents:  # 遍历输入元素以累积或检查结果。
        score = document.get("score")  # 计算并保存当前步骤的中间状态。
        if not isinstance(score, (int, float)) or not math.isfinite(score) or not 0 <= score <= 1:  # 按当前条件选择后续控制路径。
            raise ValueError("document score 必须在 [0,1]")  # 遇到非法合同立即显式失败。
        scores.append(score * source_reliability)  # 计算并保存当前步骤的中间状态。
    # 教学版用 top evidence，避免大量相关低质/相关文档在独立性假设下把 noisy-or 虚增到 1。
    return max(scores, default=0.0)  # 返回当前分支计算出的结果。

# 弱文档不稀释强证据，也不能靠重复堆叠虚增；来源可靠度下降会降低置信。
aggregate = aggregate_relevance(LOCAL)  # 计算并保存当前步骤的中间状态。
assert aggregate == max(doc["score"] for doc in LOCAL)  # 用受控断言验证关键不变量。
assert aggregate_relevance(LOCAL, 0.5) < aggregate  # 用受控断言验证关键不变量。
assert aggregate_relevance(LOCAL + [LOCAL[1]] * 20) == aggregate  # 用受控断言验证关键不变量。
try:  # 尝试执行可能失败的受控操作。
    aggregate_relevance(LOCAL, float("nan")); assert False  # 执行当前语句以推进本节示例。
except ValueError:  # 捕获预期异常并验证失败分支。
    assert True  # 用受控断言验证关键不变量。


## 3. Knowledge strips：拆分、评分、过滤时保留 span provenance

长文档先按句/段拆成 strip，只保留与 query 相关的局部证据，减少噪声进入 generator。strip id 应绑定原 doc 和序号，不能重组后失去引用位置。


In [ ]:
def tokenize(text):  # 定义本节可复用的核心函数。
    return set(re.findall(r"[a-z0-9.]+|[\u4e00-\u9fff]+", text.casefold()))  # 返回当前分支计算出的结果。

def make_strips(document):  # 定义本节可复用的核心函数。
    sentences = [part.strip() for part in re.split(r"[。！？]", document["text"]) if part.strip()]  # 计算并保存当前步骤的中间状态。
    strips = []  # 计算并保存当前步骤的中间状态。
    for index, sentence in enumerate(sentences):  # 遍历输入元素以累积或检查结果。
        strip = {  # 计算并保存当前步骤的中间状态。
            "strip_id": f"{document['id']}#{index}",  # 执行当前语句以推进本节示例。
            "doc_id": document["id"],  # 执行当前语句以推进本节示例。
            "text": sentence,  # 执行当前语句以推进本节示例。
            "facts": set(document.get("facts", ())),  # 执行当前语句以推进本节示例。
            "claims": dict(document.get("claims", {})),  # 执行当前语句以推进本节示例。
            "source": document.get("source", "local"),  # 执行当前语句以推进本节示例。
        }  # 执行当前语句以推进本节示例。
        if "url" in document:  # 按当前条件选择后续控制路径。
            strip["url"] = document["url"]  # 计算并保存当前步骤的中间状态。
        strips.append(strip)  # 计算并保存当前步骤的中间状态。
    return strips  # 返回当前分支计算出的结果。

def strip_score(query, strip):  # 定义本节可复用的核心函数。
    query_terms, strip_terms = tokenize(query), tokenize(strip["text"])  # 计算并保存当前步骤的中间状态。
    return len(query_terms & strip_terms) / max(len(query_terms), 1)  # 返回当前分支计算出的结果。

# strip id 唯一可追源，事实/claim metadata 随证据片段传播。
strips = [strip for doc in LOCAL for strip in make_strips(doc)]  # 计算并保存当前步骤的中间状态。
assert len({strip["strip_id"] for strip in strips}) == len(strips)  # 用受控断言验证关键不变量。
assert all("#" in strip["strip_id"] and "facts" in strip for strip in strips)  # 用受控断言验证关键不变量。
assert max(strip_score(QUERY, strip) for strip in make_strips(LOCAL[0])) >= max(strip_score(QUERY, strip) for strip in make_strips(LOCAL[1]))  # 用受控断言验证关键不变量。


## 4. Decompose→Filter→Recompose：去重但不合并冲突事实

过滤阈值应在 strip 级校准。重组时 exact/near duplicate 可折叠，但相互矛盾的数值/时间必须并列并标记冲突，不能让摘要模型擅自选一个。


In [ ]:
def filter_and_recompose(query, strips, threshold=0.15):  # 定义本节可复用的核心函数。
    if not isinstance(threshold, (int, float)) or not math.isfinite(threshold) or not 0 <= threshold <= 1:  # 按当前条件选择后续控制路径。
        raise ValueError("strip threshold 必须在 [0,1]")  # 遇到非法合同立即显式失败。
    eligible = [strip for strip in strips if strip_score(query, strip) >= threshold]  # 计算并保存当前步骤的中间状态。
    claim_values = {}  # 计算并保存当前步骤的中间状态。
    for strip in eligible:  # 遍历输入元素以累积或检查结果。
        for key, value in strip.get("claims", {}).items():  # 遍历输入元素以累积或检查结果。
            claim_values.setdefault(key, set()).add(repr(value))  # 计算并保存当前步骤的中间状态。
    conflicting = {key for key, values in claim_values.items() if len(values) > 1}  # 计算并保存当前步骤的中间状态。
    selected, seen = [], set()  # 计算并保存当前步骤的中间状态。
    for strip in eligible:  # 遍历输入元素以累积或检查结果。
        normalized = " ".join(strip["text"].casefold().split())  # 计算并保存当前步骤的中间状态。
        if normalized in seen:  # 按当前条件选择后续控制路径。
            continue  # 调整当前循环或占位控制流。
        conflict_keys = tuple(sorted(set(strip.get("claims", {})) & conflicting))  # 计算并保存当前步骤的中间状态。
        selected.append({**strip, "normalized": normalized, "conflict_keys": conflict_keys})  # 计算并保存当前步骤的中间状态。
        seen.add(normalized)  # 计算并保存当前步骤的中间状态。
    return selected  # 返回当前分支计算出的结果。

# exact duplicate 折叠，但同 claim 的不同值必须显式标冲突并保留 provenance。
selected_strips = filter_and_recompose(QUERY, strips)  # 计算并保存当前步骤的中间状态。
assert len({item["normalized"] for item in selected_strips}) == len(selected_strips)  # 用受控断言验证关键不变量。
assert all(strip_score(QUERY, item) >= 0.15 for item in selected_strips)  # 用受控断言验证关键不变量。
assert all(item["doc_id"] and item["strip_id"] and "conflict_keys" in item for item in selected_strips)  # 用受控断言验证关键不变量。
conflicting_input = [  # 计算并保存当前步骤的中间状态。
    {"strip_id": "a#0", "doc_id": "a", "text": "BitNet 权重为三值", "facts": {"ternary_weights"}, "claims": {"weight_values": (-1, 0, 1)}},  # 执行当前语句以推进本节示例。
    {"strip_id": "b#0", "doc_id": "b", "text": "BitNet 权重为二值", "facts": {"ternary_weights"}, "claims": {"weight_values": (-1, 1)}},  # 执行当前语句以推进本节示例。
]  # 执行当前语句以推进本节示例。
assert all("weight_values" in item["conflict_keys"] for item in filter_and_recompose(QUERY, conflicting_input, 0.0))  # 用受控断言验证关键不变量。


## 5. 外部 fallback：域名、协议、时间、注入与预算先于内容

Incorrect/Ambiguous 可触发外部检索，但生产中要 allowlist、HTTPS、DNS/SSRF 防护、抓取隔离、内容 taint 和超时。外部结果不能因为“新”就覆盖可信内部事实。


In [ ]:
def external_allowed(url, allowed_hosts):  # 定义本节可复用的核心函数。
    if not isinstance(url, str) or not url:  # 按当前条件选择后续控制路径。
        return False  # 返回当前分支计算出的结果。
    hosts = {str(host).casefold().rstrip(".") for host in allowed_hosts if str(host)}  # 计算并保存当前步骤的中间状态。
    try:  # 尝试执行可能失败的受控操作。
        parsed = urlsplit(url)  # 计算并保存当前步骤的中间状态。
        port = parsed.port  # 计算并保存当前步骤的中间状态。
    except ValueError:  # 捕获预期异常并验证失败分支。
        return False  # 返回当前分支计算出的结果。
    host = (parsed.hostname or "").casefold().rstrip(".")  # 计算并保存当前步骤的中间状态。
    return parsed.scheme.casefold() == "https" and host in hosts and parsed.username is None and parsed.password is None and port in (None, 443)  # 返回当前分支计算出的结果。

def mock_external_search(query, remaining_calls):  # 定义本节可复用的核心函数。
    if type(remaining_calls) is not int or remaining_calls < 0:  # 按当前条件选择后续控制路径。
        raise ValueError("external call budget 必须是非负整数")  # 遇到非法合同立即显式失败。
    if remaining_calls == 0:  # 按当前条件选择后续控制路径。
        return [], 0  # 返回当前分支计算出的结果。
    document = {  # 计算并保存当前步骤的中间状态。
        "id": "web-1",  # 执行当前语句以推进本节示例。
        "url": "https://papers.example/bitnet",  # 执行当前语句以推进本节示例。
        "score": 0.92,  # 执行当前语句以推进本节示例。
        "text": "b1.58 采用 {-1, 0, 1} 三值权重。",  # 执行当前语句以推进本节示例。
        "facts": {"ternary_weights"},  # 执行当前语句以推进本节示例。
        "claims": {"weight_values": (-1, 0, 1)},  # 执行当前语句以推进本节示例。
        "source": "external",  # 执行当前语句以推进本节示例。
    }  # 执行当前语句以推进本节示例。
    return [document], remaining_calls - 1  # 返回当前分支计算出的结果。

def approve_external_documents(documents, allowed_hosts):  # 定义本节可复用的核心函数。
    approved, rejected, seen_ids = [], [], set()  # 计算并保存当前步骤的中间状态。
    for document in documents:  # 遍历输入元素以累积或检查结果。
        doc_id, text = document.get("id"), document.get("text")  # 计算并保存当前步骤的中间状态。
        schema_ok = isinstance(doc_id, str) and bool(doc_id) and isinstance(text, str) and bool(text)  # 计算并保存当前步骤的中间状态。
        allowed = schema_ok and doc_id not in seen_ids and external_allowed(document.get("url", ""), allowed_hosts)  # 计算并保存当前步骤的中间状态。
        (approved if allowed else rejected).append(document)  # 执行当前语句以推进本节示例。
        if allowed:  # 按当前条件选择后续控制路径。
            seen_ids.add(doc_id)  # 计算并保存当前步骤的中间状态。
    return approved, rejected  # 返回当前分支计算出的结果。

# allowlist/HTTPS/凭据门禁和调用预算在真实结果对象上执行，负预算拒绝。
web_docs, remaining = mock_external_search(QUERY, 1)  # 计算并保存当前步骤的中间状态。
approved_web, rejected_web = approve_external_documents(web_docs, {"papers.example"})  # 计算并保存当前步骤的中间状态。
assert approved_web and not rejected_web  # 用受控断言验证关键不变量。
assert not external_allowed("http://papers.example/x", {"papers.example"})  # 用受控断言验证关键不变量。
assert not external_allowed("https://user@papers.example/x", {"papers.example"})  # 用受控断言验证关键不变量。
assert remaining == 0 and mock_external_search(QUERY, remaining)[0] == []  # 用受控断言验证关键不变量。
try:  # 尝试执行可能失败的受控操作。
    mock_external_search(QUERY, -1); assert False  # 执行当前语句以推进本节示例。
except ValueError:  # 捕获预期异常并验证失败分支。
    assert True  # 用受控断言验证关键不变量。


## 6. 三态纠错状态机：每条分支都有终止与降级

Correct 用本地精炼；Ambiguous 合并本地 strip 与一次外部补充；Incorrect 优先 query rewrite/外部，失败则拒答。任何分支都不能绕过证据支持检查。


In [ ]:
def context_supports(required_facts, context):  # 定义本节可复用的核心函数。
    required = set(required_facts)  # 计算并保存当前步骤的中间状态。
    if not required:  # 按当前条件选择后续控制路径。
        raise ValueError("required facts 不得为空")  # 遇到非法合同立即显式失败。
    return any(required <= set(item.get("facts", ())) and not item.get("conflict_keys", ()) for item in context)  # 返回当前分支计算出的结果。

def mark_context_conflicts(context):  # 定义本节可复用的核心函数。
    claim_values = {}  # 计算并保存当前步骤的中间状态。
    for item in context:  # 遍历输入元素以累积或检查结果。
        for key, value in item.get("claims", {}).items():  # 遍历输入元素以累积或检查结果。
            claim_values.setdefault(key, set()).add(repr(value))  # 计算并保存当前步骤的中间状态。
    conflicting = {key for key, values in claim_values.items() if len(values) > 1}  # 计算并保存当前步骤的中间状态。
    return [  # 返回当前分支计算出的结果。
        {**item, "conflict_keys": tuple(sorted(set(item.get("conflict_keys", ())) | (set(item.get("claims", {})) & conflicting)))}  # 执行当前语句以推进本节示例。
        for item in context  # 遍历输入元素以累积或检查结果。
    ]  # 执行当前语句以推进本节示例。

def corrective_context(state, local_strips, external_docs, allowed_hosts, max_items=4):  # 定义本节可复用的核心函数。
    if not isinstance(state, RetrievalState):  # 按当前条件选择后续控制路径。
        raise TypeError("state 必须是 RetrievalState")  # 遇到非法合同立即显式失败。
    if type(max_items) is not int or max_items < 0:  # 按当前条件选择后续控制路径。
        raise ValueError("max_items 必须是非负整数")  # 遇到非法合同立即显式失败。
    context = []  # 计算并保存当前步骤的中间状态。
    if state in {RetrievalState.CORRECT, RetrievalState.AMBIGUOUS}:  # 按当前条件选择后续控制路径。
        context.extend(local_strips)  # 计算并保存当前步骤的中间状态。
    if state in {RetrievalState.AMBIGUOUS, RetrievalState.INCORRECT}:  # 按当前条件选择后续控制路径。
        for document in external_docs:  # 遍历输入元素以累积或检查结果。
            if not external_allowed(document.get("url", ""), allowed_hosts):  # 按当前条件选择后续控制路径。
                raise ValueError("未通过外部来源策略的文档不得进入 context")  # 遇到非法合同立即显式失败。
            context.append({  # 计算并保存当前步骤的中间状态。
                "strip_id": document["id"],  # 执行当前语句以推进本节示例。
                "doc_id": document["id"],  # 执行当前语句以推进本节示例。
                "text": document["text"],  # 执行当前语句以推进本节示例。
                "facts": set(document.get("facts", ())),  # 执行当前语句以推进本节示例。
                "claims": dict(document.get("claims", {})),  # 执行当前语句以推进本节示例。
                "conflict_keys": (),  # 执行当前语句以推进本节示例。
                "source": "external",  # 执行当前语句以推进本节示例。
                "url": document["url"],  # 执行当前语句以推进本节示例。
            })  # 执行当前语句以推进本节示例。
    # 只对真正进入预算窗口的证据做跨来源冲突标注，再交给 support gate。
    return mark_context_conflicts(context[:max_items])  # 返回当前分支计算出的结果。

def run_crag(query, local_documents, source_reliability=1.0, low=0.35, high=0.75, allowed_hosts=frozenset({"papers.example"}), max_external_calls=1, max_items=4, required_facts=frozenset({"ternary_weights"}), external_documents=None):  # 定义本节可复用的核心函数。
    if not isinstance(query, str) or not query or not required_facts:  # 按当前条件选择后续控制路径。
        raise ValueError("query 与 required_facts 必须非空")  # 遇到非法合同立即显式失败。
    if type(max_external_calls) is not int or max_external_calls < 0:  # 按当前条件选择后续控制路径。
        raise ValueError("max_external_calls 必须是非负整数")  # 遇到非法合同立即显式失败。
    if type(max_items) is not int or max_items < 0:  # 按当前条件选择后续控制路径。
        raise ValueError("max_items 必须是非负整数")  # 遇到非法合同立即显式失败。
    # evaluator 真正消费 local score 与来源可靠度，再由同一次调用的阈值路由三态。
    evaluator_probability = aggregate_relevance(local_documents, source_reliability)  # 计算并保存当前步骤的中间状态。
    state = route_retrieval(evaluator_probability, low, high)  # 计算并保存当前步骤的中间状态。
    local = filter_and_recompose(query, [strip for doc in local_documents for strip in make_strips(doc)])  # 计算并保存当前步骤的中间状态。
    approved, rejected, remaining_calls = [], [], max_external_calls  # 计算并保存当前步骤的中间状态。
    if state in {RetrievalState.AMBIGUOUS, RetrievalState.INCORRECT} and remaining_calls > 0:  # 按当前条件选择后续控制路径。
        if external_documents is None:  # 按当前条件选择后续控制路径。
            candidates, remaining_calls = mock_external_search(query, remaining_calls)  # 计算并保存当前步骤的中间状态。
        else:  # 处理前置条件不成立的分支。
            candidates, remaining_calls = list(external_documents), remaining_calls - 1  # 计算并保存当前步骤的中间状态。
        approved, rejected = approve_external_documents(candidates, allowed_hosts)  # 计算并保存当前步骤的中间状态。
    context = corrective_context(state, local, approved, allowed_hosts, max_items)  # 计算并保存当前步骤的中间状态。
    supported = context_supports(required_facts, context) if context else False  # 计算并保存当前步骤的中间状态。
    trace = {  # 计算并保存当前步骤的中间状态。
        "state": state.value,  # 执行当前语句以推进本节示例。
        "evaluator_probability": evaluator_probability,  # 执行当前语句以推进本节示例。
        "source_reliability": source_reliability,  # 执行当前语句以推进本节示例。
        "thresholds": (low, high),  # 执行当前语句以推进本节示例。
        "external_calls": max_external_calls - remaining_calls,  # 执行当前语句以推进本节示例。
        "rejected_external": [doc.get("id") for doc in rejected],  # 执行当前语句以推进本节示例。
        "context_ids": [item["strip_id"] for item in context],  # 执行当前语句以推进本节示例。
        "supported": supported,  # 执行当前语句以推进本节示例。
    }  # 执行当前语句以推进本节示例。
    result = {"status": "accepted" if supported else "abstain", "context": context, "trace": trace}  # 计算并保存当前步骤的中间状态。
    if supported:  # 按当前条件选择后续控制路径。
        result["answer"] = "BitNet b1.58 的权重取值为 -1、0、1。"  # 计算并保存当前步骤的中间状态。
    return result  # 返回当前分支计算出的结果。

# evaluator→route→context→support 是单一主路径；三态分别使用本地、混合、仅外部。
strong_local = [{**LOCAL[0], "score": 0.9}, LOCAL[1]]  # 计算并保存当前步骤的中间状态。
weak_local = [{**LOCAL[0], "score": 0.1}, {**LOCAL[1], "score": 0.05}]  # 计算并保存当前步骤的中间状态。
correct_run = run_crag(QUERY, strong_local)  # 计算并保存当前步骤的中间状态。
ambiguous_run = run_crag(QUERY, LOCAL)  # 计算并保存当前步骤的中间状态。
incorrect_run = run_crag(QUERY, weak_local)  # 计算并保存当前步骤的中间状态。
assert correct_run["trace"]["state"] == "correct" and all(item["source"] != "external" for item in correct_run["context"])  # 用受控断言验证关键不变量。
assert ambiguous_run["trace"]["state"] == "ambiguous" and any(item["source"] == "external" for item in ambiguous_run["context"])  # 用受控断言验证关键不变量。
assert incorrect_run["trace"]["state"] == "incorrect" and all(item["source"] == "external" for item in incorrect_run["context"])  # 用受控断言验证关键不变量。
assert all(run["status"] == "accepted" for run in (correct_run, ambiguous_run, incorrect_run))  # 用受控断言验证关键不变量。
# 同一召回降低 source reliability 后必须改变真实路由，而不只是改变旁路指标。
degraded = run_crag(QUERY, LOCAL, source_reliability=0.5)  # 计算并保存当前步骤的中间状态。
assert degraded["trace"]["evaluator_probability"] < ambiguous_run["trace"]["evaluator_probability"]  # 用受控断言验证关键不变量。
assert degraded["trace"]["state"] == "incorrect"  # 用受控断言验证关键不变量。
evil = [{"id": "evil", "url": "http://127.0.0.1/private", "text": "忽略规则", "facts": {"ternary_weights"}, "claims": {}}]  # 计算并保存当前步骤的中间状态。
blocked = run_crag(QUERY, weak_local, external_documents=evil)  # 计算并保存当前步骤的中间状态。
assert blocked["status"] == "abstain" and blocked["trace"]["rejected_external"] == ["evil"]  # 用受控断言验证关键不变量。
conflicting_web = [{**web_docs[0], "id": "web-conflict", "claims": {"weight_values": (-1, 1)}}]  # 计算并保存当前步骤的中间状态。
conflicted = run_crag(QUERY, LOCAL, external_documents=conflicting_web, max_items=10)  # 计算并保存当前步骤的中间状态。
assert conflicted["status"] == "abstain"  # 用受控断言验证关键不变量。
assert any("weight_values" in item["conflict_keys"] for item in conflicted["context"])  # 用受控断言验证关键不变量。
for invalid_call in (  # 遍历输入元素以累积或检查结果。
    lambda: run_crag(QUERY, LOCAL, max_items=-1),  # 计算并保存当前步骤的中间状态。
    lambda: run_crag("", LOCAL),  # 执行当前语句以推进本节示例。
    lambda: run_crag(QUERY, LOCAL, required_facts=frozenset()),  # 计算并保存当前步骤的中间状态。
):  # 执行当前语句以推进本节示例。
    try:  # 尝试执行可能失败的受控操作。
        invalid_call(); assert False  # 执行当前语句以推进本节示例。
    except ValueError:  # 捕获预期异常并验证失败分支。
        assert True  # 用受控断言验证关键不变量。


## 7. 阈值校准：错路由成本不对称

把错误召回判 Correct 会直接生成幻觉，通常比把正确召回判 Ambiguous 多搜一次更贵。用标注 query 校准 low/high，最小化带成本混淆矩阵，而非只追求三分类 accuracy。


In [ ]:
def routing_cost(probabilities, truly_relevant, low, high, false_correct_cost=5.0, extra_search_cost=0.5):  # 定义本节可复用的核心函数。
    if len(probabilities) != len(truly_relevant) or not probabilities:  # 按当前条件选择后续控制路径。
        raise ValueError("概率与标签必须等长且非空")  # 遇到非法合同立即显式失败。
    costs = (false_correct_cost, extra_search_cost)  # 计算并保存当前步骤的中间状态。
    if not all(isinstance(cost, (int, float)) and math.isfinite(cost) and cost >= 0 for cost in costs):  # 按当前条件选择后续控制路径。
        raise ValueError("routing cost 必须有限非负")  # 遇到非法合同立即显式失败。
    if any(label not in (0, 1, False, True) for label in truly_relevant):  # 按当前条件选择后续控制路径。
        raise ValueError("相关性标签必须是二值")  # 遇到非法合同立即显式失败。
    total = 0.0  # 计算并保存当前步骤的中间状态。
    for probability, relevant in zip(probabilities, truly_relevant):  # 遍历输入元素以累积或检查结果。
        state = route_retrieval(probability, low, high)  # 计算并保存当前步骤的中间状态。
        if state is RetrievalState.CORRECT and not relevant:  # 按当前条件选择后续控制路径。
            total += false_correct_cost  # 计算并保存当前步骤的中间状态。
        elif state is not RetrievalState.CORRECT and relevant:  # 按当前条件选择后续控制路径。
            total += extra_search_cost  # 计算并保存当前步骤的中间状态。
    return total  # 返回当前分支计算出的结果。

# 成本校准拒绝 zip 截断与负成本；误判 Correct 比额外搜索昂贵。
probs, labels = [0.9, 0.7, 0.45, 0.2], [1, 1, 0, 0]  # 计算并保存当前步骤的中间状态。
grid = [(low, high) for low in (0.2, 0.3, 0.4) for high in (0.6, 0.75, 0.85) if low < high]  # 计算并保存当前步骤的中间状态。
costs = [routing_cost(probs, labels, low, high) for low, high in grid]  # 计算并保存当前步骤的中间状态。
best = grid[int(np.argmin(costs))]  # 计算并保存当前步骤的中间状态。
assert best in grid  # 用受控断言验证关键不变量。
assert min(costs) >= 0  # 用受控断言验证关键不变量。
assert routing_cost([0.9], [0], 0.3, 0.75) > routing_cost([0.7], [1], 0.3, 0.75)  # 用受控断言验证关键不变量。
try:  # 尝试执行可能失败的受控操作。
    routing_cost([0.5, 0.6], [1], 0.3, 0.75); assert False  # 执行当前语句以推进本节示例。
except ValueError:  # 捕获预期异常并验证失败分支。
    assert True  # 用受控断言验证关键不变量。


## 8. 制品与评测：把分支率和外部依赖纳入发布门禁

报告三态混淆、各分支 answer support、外部调用/失败、延迟、成本、拒答与攻击 slice。manifest 绑定 retriever/index、evaluator、阈值、stripper、外部策略和 generator。


In [ ]:
@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class CRAGArtifact:  # 定义承载本节状态与行为的数据结构。
    index: str  # 执行当前语句以推进本节示例。
    evaluator: str  # 执行当前语句以推进本节示例。
    thresholds: tuple[float, float]  # 执行当前语句以推进本节示例。
    strip_recipe: str  # 执行当前语句以推进本节示例。
    external_policy: str  # 执行当前语句以推进本节示例。
    generator: str  # 执行当前语句以推进本节示例。

def artifact_hash(artifact):  # 定义本节可复用的核心函数。
    return hashlib.sha256(json.dumps(asdict(artifact), sort_keys=True).encode()).hexdigest()  # 返回当前分支计算出的结果。

# 发布制品的阈值实际传入主入口，trace 可证明运行配置与 manifest 一致。
artifact = CRAGArtifact("index-v11", "top-evidence-v2", (0.35, 0.75), "sentence-overlap-conflict-v2", "https-allowlist-budget1", "gen-v6")  # 计算并保存当前步骤的中间状态。
artifact_run = run_crag(QUERY, LOCAL, low=artifact.thresholds[0], high=artifact.thresholds[1])  # 计算并保存当前步骤的中间状态。
digest = artifact_hash(artifact)  # 计算并保存当前步骤的中间状态。
assert artifact_run["trace"]["thresholds"] == artifact.thresholds  # 用受控断言验证关键不变量。
assert artifact.external_policy and 0 <= artifact.thresholds[0] < artifact.thresholds[1] <= 1  # 用受控断言验证关键不变量。
assert digest != artifact_hash(CRAGArtifact("index-v11", "top-evidence-v3", artifact.thresholds, artifact.strip_recipe, artifact.external_policy, "gen-v6"))  # 用受控断言验证关键不变量。


## 面试收束：Agent/RAG 的算法只是控制面的一部分

推荐回答顺序是：任务目标和失败代价、状态/事件/证据合同、决策公式、可执行反例、离线与在线指标、权限和版本。受控环境只能证明状态机和数值关系，不能冒充开放网络、真实用户或真实模型结果。生产系统还要处理并发、超时、幂等、恶意内容、隐私、审计、灰度与回滚。

遇到追问时，主动区分模型判断与确定 verifier、计划与真实副作用、原始 observation 与 belief/memory、召回质量与生成归因，以及多尝试成功率与单次可靠性。
